# Lista 5

## Pandas

(5pkt + 2pkt)

Na liście znajdują się 3 zadania, kolejno za 2,1,2. Po rozwiązaniu zadania pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Zadanie domowe oznaczone jest ⭐️. Za wykonanie zadania otrzymasz 2pkt.

Korzystamy z danych z wykładu: `./data/student_exam_scores.csv` oraz dziennika sesji nauki `./data/study_sessions.csv`.

In [1]:
import numpy as np
import pandas as pd
import os, sys, glob, shutil, subprocess

DATA_PATH = './data/student_exam_scores.csv'

# Pobranie danych z Kaggle (tylko za pierwszym razem): https://www.kaggle.com/datasets/saadaliyaseen/analyzing-student-academic-trends
if not os.path.exists(DATA_PATH):
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
        import kagglehub
    folder = kagglehub.dataset_download('saadaliyaseen/analyzing-student-academic-trends')
    csv_file = glob.glob(os.path.join(folder, '**', '*.csv'), recursive=True)[0]
    os.makedirs('./data', exist_ok=True)
    shutil.copy(csv_file, DATA_PATH)
    print('Pobrano:', os.path.basename(csv_file), '->', DATA_PATH)

LOG_PATH = './data/study_sessions.csv'

# Jeśli nie masz pliku z dziennikiem sesji z wykładu — ta komórka go wygeneruje (ten sam kod co na wykładzie).
if not os.path.exists(LOG_PATH):
    rng = np.random.default_rng(42)
    students = pd.read_csv(DATA_PATH)
    n_sessions = np.maximum(1, (students['hours_studied'] * rng.uniform(40, 60, len(students))).round()).astype(int)
    sid   = np.repeat(students['student_id'].to_numpy(), n_sessions)
    sleep = np.repeat(students['sleep_hours'].to_numpy(), n_sessions)
    n = len(sid)
    p_night = np.clip((8 - sleep) / 4, 0.05, 0.8)
    night = rng.random(n) < p_night
    hour = np.where(night, rng.choice([22, 23, 0, 1, 2, 3], n), rng.integers(8, 22, n))
    day = pd.to_datetime('2025-10-01') + pd.to_timedelta(rng.integers(0, 120, n), unit='D')
    start = day + pd.to_timedelta(hour, unit='h') + pd.to_timedelta(rng.integers(0, 60, n), unit='min')
    log = pd.DataFrame({
        'student_id': sid,
        'start_time': start,
        'subject': rng.choice(['math', 'physics', 'programming', 'statistics'], n, p=[0.35, 0.2, 0.3, 0.15]),
        'duration_min': rng.gamma(3, 15, n).round().astype(int) + 5,
    }).sort_values('start_time')
    log.insert(0, 'session_id', np.arange(n))
    log.to_csv(LOG_PATH, index=False)

print('OK')


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.37k/2.37k [00:00<00:00, 1.48MB/s]

Extracting files...
Pobrano: student_exam_scores.csv -> ./data/student_exam_scores.csv
OK


#### Zadanie 1

Każdy podpunkt rozwiąż **jednym** wywołaniem funkcji (ewentualnie z prostym warunkiem w nawiasach).

a) Wczytaj `LOG_PATH` do zmiennej `sessions` tak, żeby kolumna `start_time` miała typ daty.

b) Ilu studentów ma frekwencję (`attendance_percent`) poniżej 60%?

c) Wyświetl 5 studentów z najwyższym `exam_score` (tylko kolumny `student_id` i `exam_score`).

d) Ile sesji odbyło się z każdego przedmiotu (`subject`)?

e) Dodaj do `sessions` kolumnę `weekday` z nazwą dnia tygodnia, w którym zaczęła się sesja.


In [2]:
df = pd.read_csv(DATA_PATH)
df.head()

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
0,S001,8.0,8.8,72.1,45,30.2
1,S002,1.3,8.6,60.7,55,25.0
2,S003,4.0,8.2,73.7,86,35.8
3,S004,3.5,4.8,95.1,66,34.0
4,S005,9.1,6.4,89.8,71,40.3


In [4]:
# a)
sessions = pd.read_csv(LOG_PATH, parse_dates=['start_time'])
sessions.head()

,session_id,student_id,start_time,subject,duration_min
0,0,S113,2025-10-01 00:01:00,programming,55
1,1,S116,2025-10-01 00:02:00,statistics,26
2,2,S008,2025-10-01 00:03:00,statistics,50
3,3,S075,2025-10-01 00:04:00,physics,16
4,4,S154,2025-10-01 00:05:00,math,47


In [7]:
# b)
print(len(df[df["attendance_percent"] < 60.0]))

37


In [9]:
# c)
df[["student_id", "exam_score"]].sort_values("exam_score", ascending=False).head()

,student_id,exam_score
86,S087,51.3
83,S084,48.9
60,S061,48.6
120,S121,47.9
71,S072,47.9


In [16]:
# d)
sessions['subject'].value_counts()

subject
math           22132
programming    19076
physics        12616
statistics      9488
Name: count, dtype: int64

In [18]:
# e)
sessions["weekday"] = sessions["start_time"].dt.day_name()
sessions.head()

,session_id,student_id,start_time,subject,duration_min,weekday
0,0,S113,2025-10-01 00:01:00,programming,55,Wednesday
1,1,S116,2025-10-01 00:02:00,statistics,26,Wednesday
2,2,S008,2025-10-01 00:03:00,statistics,50,Wednesday
3,3,S075,2025-10-01 00:04:00,physics,16,Wednesday
4,4,S154,2025-10-01 00:05:00,math,47,Wednesday


#### Zadanie 2

W każdym podpunkcie połącz 2–3 funkcje.

a) Komórka poniżej tworzy `dfm` — kopię danych z brakami. Usuń wiersze, w których brakuje `exam_score` (tylko te!),
a braki w `sleep_hours` uzupełnij medianą tej kolumny. Pokaż liczbę braków w każdej kolumnie przed i po. (`isna`, `dropna`, `fillna`)

In [19]:
# a) dane z brakami — uruchom
rng = np.random.default_rng(5)
dfm = df[['student_id', 'hours_studied', 'sleep_hours', 'exam_score']].copy()
dfm.loc[rng.choice(dfm.index, 30, replace=False), 'sleep_hours'] = np.nan
dfm.loc[rng.choice(dfm.index, 8, replace=False), 'exam_score'] = np.nan

In [26]:
# a)
print(dfm.isna().sum())
df_clean = dfm.dropna(subset=["exam_score"]).fillna({"sleep_hours": dfm["sleep_hours"].mean()}).head()
print(df_clean.isna().sum())

student_id        0
hours_studied     0
sleep_hours      30
exam_score        8
dtype: int64
student_id       0
hours_studied    0
sleep_hours      0
exam_score       0
dtype: int64


#### Zadanie 3 — LLM wygenerował Ci kod, sprawdź które fragmenty są poprawne, a które błędne — napraw błędne

Poprosiliśmy model językowy o kod realizujący cele opisane przy każdym fragmencie. Część fragmentów jest poprawna, część — nie.
Błędy bywają głośne (wyjątek) i **ciche** (kod działa, ale wynik jest zły).

Dla każdego fragmentu:
1. uruchom go i zdecyduj: **OK** czy **BŁĄD**,
2. jeśli BŁĄD — napisz jednym zdaniem, na czym polega, i wpisz poprawioną wersję w komórce pod nim.

Każda komórka zaczyna się od wczytania danych od nowa, więc fragmenty nie wpływają na siebie nawzajem.

**Fragment 1.** Cel: Studenci, którzy śpią mniej niż 6 h i mają frekwencję poniżej 70%.

In [27]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)

# kod wygenerowany przez LLM
wynik = df[df['sleep_hours'] < 6 and df['attendance_percent'] < 70]
wynik

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [28]:
# Fragment 1: OK / BŁĄD? Uzasadnienie i poprawka:

# blad ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

df = pd.read_csv(DATA_PATH)

wynik = df[(df['sleep_hours'] < 6) & (df['attendance_percent'] < 70)] # inaczej napisana maska
wynik

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
5,S006,8.4,5.1,58.5,75,35.7
19,S020,1.1,5.5,53.6,65,17.1
30,S031,9.9,4.4,55.3,67,34.5
35,S036,7.1,4.6,52.5,84,32.0
62,S063,7.1,5.8,63.6,46,27.1
63,S064,8.5,4.8,66.0,93,36.1
69,S070,3.9,4.0,53.2,59,23.9
81,S082,3.9,5.0,50.3,80,32.4
90,S091,2.2,4.0,61.0,70,27.3
93,S094,5.6,5.3,51.5,61,29.1


**Fragment 2.** Cel: Średni exam_score w trzech grupach według czasu nauki.

In [43]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)
print(df.isna().sum()) # sprawdzenie braków danych

print(df.head())
# kod wygenerowany przez LLM
df.groupby(pd.cut(df['hours_studied'], bins=3), observed=True)['exam_score'].mean()

student_id            0
hours_studied         0
sleep_hours           0
attendance_percent    0
previous_scores       0
exam_score            0
dtype: int64
  student_id  hours_studied  sleep_hours  attendance_percent  previous_scores  \
0       S001            8.0          8.8                72.1               45   
1       S002            1.3          8.6                60.7               55   
2       S003            4.0          8.2                73.7               86   
3       S004            3.5          4.8                95.1               66   
4       S005            9.1          6.4                89.8               71   

   exam_score  
0        30.2  
1        25.0  
2        35.8  
3        34.0  
4        40.3  


hours_studied
(0.989, 4.667]    28.480000
(4.667, 8.333]    34.137500
(8.333, 12.0]     40.495082
Name: exam_score, dtype: float64

In [ ]:
# Fragment 2: OK / BŁĄD? Uzasadnienie i poprawka:

# Wydaje sie git

0     30.2
1     25.0
2     35.8
3     34.0
4     40.3
5     35.7
6     37.9
7     18.3
8     34.7
9     24.7
11    35.1
14    41.1
15    34.1
18    35.6
20    46.0
Name: exam_score, dtype: float64

**Fragment 3.** Cel: Kolumna grade: 'A' dla exam_score > 35, w przeciwnym razie 'B'.

In [ ]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)

# kod wygenerowany przez LLM
df['grade'] = 'B'
df[df['exam_score'] > 35]['grade'] = 'A'
df['grade'].value_counts()

/var/folders/y3/66k6vqmd0qb4ylndn4_qhys00000gn/T/ipykernel_96274/10764233.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df['exam_score'] > 35]['grade'] = 'A'


In [ ]:
# Fragment 3: OK / BŁĄD? Uzasadnienie i poprawka:

# wczytanie danych
df = pd.read_csv(DATA_PATH)

# kod wygenerowany przez LLM
df['grade'] = 'B'
df.loc[df['exam_score'] > 35, 'grade'] = 'A' # trzeba użyć loc, bo iunaczej SettingWithCopyWarning

**Fragment 4.** Cel: Wiersze z brakującą wartością sleep_hours w dfm.

In [51]:
# wczytanie danych
dfm = pd.read_csv(DATA_PATH)
dfm.loc[dfm.sample(20, random_state=0).index, 'sleep_hours'] = np.nan

# kod wygenerowany przez LLM
dfm[dfm['sleep_hours'] == np.nan]

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score


In [ ]:
# Fragment 4: OK / BŁĄD? Uzasadnienie i poprawka:

dfm[dfm['sleep_hours'].isna()] # == nie dziala dla nan


,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
5,S006,8.4,NaN,58.5,75,35.7
7,S008,2.0,NaN,75.8,55,18.3
12,S013,1.3,NaN,83.6,71,31.2
18,S019,9.9,NaN,92.5,54,35.6
33,S034,11.7,NaN,97.9,61,42.7
37,S038,7.8,NaN,92.4,57,38.3
61,S062,8.0,NaN,64.3,58,36.1
74,S075,8.2,NaN,71.0,57,33.5
80,S081,7.2,NaN,69.0,48,30.8
98,S099,11.7,NaN,73.4,70,39.9


**Fragment 5.** Cel: Dołącz do df łączny czas nauki każdego studenta (kolumna total_min).

In [53]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)
sessions = pd.read_csv(LOG_PATH, parse_dates=['start_time'])

# kod wygenerowany przez LLM
total = sessions.groupby('student_id')['duration_min'].sum().rename('total_min')
df2 = df.join(total, on='student_id')
df2.head()

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score,total_min
0,S001,8.0,8.8,72.1,45,30.2,22306
1,S002,1.3,8.6,60.7,55,25.0,3354
2,S003,4.0,8.2,73.7,86,35.8,11851
3,S004,3.5,4.8,95.1,66,34.0,9636
4,S005,9.1,6.4,89.8,71,40.3,18087


In [ ]:
# Fragment 5: OK / BŁĄD? Uzasadnienie i poprawka:

# Wydaje się git

**Fragment 6.** Cel: Posortuj studentów malejąco według exam_score i pokaż trzech najlepszych.

In [59]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)

# kod wygenerowany przez LLM
df.sort_values('exam_score', ascending=False).head(3)
#df.head(3)

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
86,S087,12.0,8.2,71.0,95,51.3
83,S084,10.9,5.3,87.7,88,48.9
60,S061,11.9,8.4,53.9,78,48.6


In [ ]:
# Fragment 6: OK / BŁĄD? Uzasadnienie i poprawka:

# wczytanie danych
df = pd.read_csv(DATA_PATH)

# kod wygenerowany przez LLM
df.sort_values('exam_score', ascending=False).head(3) # zmiany nie byly inplace

**Fragment 7.** Cel: Średni exam_score studentów, którzy mieli co najmniej jedną sesję z programowania.

In [60]:
# wczytanie danych
df = pd.read_csv(DATA_PATH)
sessions = pd.read_csv(LOG_PATH, parse_dates=['start_time'])

# kod wygenerowany przez LLM
prog = sessions[sessions['subject'] == 'programming']
df.merge(prog, on='student_id')['exam_score'].mean()

np.float64(36.640721325225414)

In [62]:
# Fragment 7: OK / BŁĄD? Uzasadnienie i poprawka:

prog = sessions[sessions['subject'] == 'programming'].drop_duplicates(subset='student_id', keep='first')
df.merge(prog, on='student_id')['exam_score'].mean()

np.float64(33.955)